# Wireless Coverage — Part 3: H3 Gridding of DEM / DSM / CHM Surfaces

Prepare the LiDAR-derived elevation surfaces from Part 2 for global-grid analysis by
H3-indexing each surface. DEM and DSM elevation bands are extracted as contiguous isoband
polygons (`rst_isoband`) and indexed to H3 cells with the Databricks product
`h3_try_coverash3`. The Canopy Height Model is aggregated to the maximum canopy height per
H3 cell (`gbx_rst_h3_rastertogridmax`). Gaps over land are filled with `h3_cellfill`
(k=1, IDW). The resulting H3 tables are prepared for downstream coverage analysis — antenna
siting and line-of-sight in Part 4.

> **Runtime.** Runs on **Databricks Serverless environment 5** — the lightweight tier (pure
> Python/PySpark, no JAR or GDAL init script). Set `DEMO=True` (default) for a Golden Gate
> Park sub-AOI (~25 tiles) to iterate quickly; `DEMO=False` for full San Francisco.

---
**Last Update:** 2026-10-01


In [ ]:
%pip install --quiet --disable-pip-version-check --no-deps --force-reinstall \
  "geobrix[light_env5,vizx,overture] @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix-0.5.2-py3-none-any.whl"
%pip install --quiet \
  "geobrix[light_env5,vizx,overture] @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix-0.5.2-py3-none-any.whl"


In [ ]:
%restart_python


In [ ]:
import warnings, glob, math
warnings.filterwarnings("ignore")
from pyspark.sql import functions as F
from pyspark.databricks.sql import functions as DBF
from shapely import wkb as _wkb
from shapely.geometry import box as _box
from shapely.ops import unary_union

from databricks.labs.gbx.pyrx import functions as rx
from databricks.labs.gbx.pygx import functions as gx
from databricks.labs.gbx.pygx import _cellfill
from databricks.labs.gbx.ds.register import register
from databricks.labs.gbx.vizx import (
    plot_static, plot_point_cloud,
    raster_layer, vector_layer, point_cloud_layer, cells_as_gdf,
)
from databricks.labs.gbx.sample.overture import OvertureClient

rx.register(spark)
gx.register(spark)
register(spark)


In [ ]:
# --- source surfaces (produced by nb2) ---
CATALOG      = "geospatial_docs"
SCHEMA       = "geobrix"
SRC_PREFIX   = "wc_surface"            # wc_surface_{dsm,dtm,chm}
OUT_PREFIX   = "wc_h3"                 # wc_h3_{dem,dsm,chm}
LAZ_DIR      = "/Volumes/geospatial_docs/wireless_coverage/data/lidar/sf/laz"
LAND_DIR     = "/Volumes/geospatial_docs/geobrix/sample-data/geobrix-examples/sf/overture-water-bay"

# --- H3 grid ---
H3_RESOLUTION = 9                      # ≈174 m edge; set 8 (≈460 m) for a fast coarse pass
# 10 elevation bands × 30 m — matches SF DTM/DSM range (~0–280 m).
BREAKS_M     = [float(b) for b in range(0, 301, 30)]

# --- scope: DEMO=True limits to Golden Gate Park ±2 km tiles (~25 tiles) ---
DEMO         = True
SF_AOI       = (-122.55, 37.70, -122.35, 37.85)   # full San Francisco lon/lat BBOX
TILE_M       = 1024.0                  # matches nb1/nb2 tile size

FORCE_RELOAD = False                   # True → rebuild H3 tables; False → reuse


In [ ]:
def _persist(df, name):
    """Write to ``CATALOG.SCHEMA.OUT_PREFIX_<name>`` and return it read back.
    FORCE_RELOAD rebuilds; otherwise an existing table is reused."""
    tbl = f"{CATALOG}.{SCHEMA}.{OUT_PREFIX}_{name}"
    if FORCE_RELOAD or not spark.catalog.tableExists(tbl):
        df.write.mode("overwrite").saveAsTable(tbl)
    return spark.table(tbl)


In [ ]:
dsm = spark.table(f"{CATALOG}.{SCHEMA}.{SRC_PREFIX}_dsm")
dtm = spark.table(f"{CATALOG}.{SCHEMA}.{SRC_PREFIX}_dtm")
chm = spark.table(f"{CATALOG}.{SCHEMA}.{SRC_PREFIX}_chm")
print(f"Source surfaces: DSM {dsm.count()} | DTM {dtm.count()} | CHM {chm.count()} tiles")

# Confirm CRS (all should be EPSG:3857 — requires rst_transform before H3 indexing).
sample_srid = dsm.select(F.expr("gbx_rst_srid(dsm)").alias("srid")).first()["srid"]
assert sample_srid == 3857, f"Expected EPSG:3857, got {sample_srid}"
print(f"Surface CRS confirmed: EPSG:{sample_srid}")


In [ ]:
laz_files = sorted(glob.glob(f"{LAZ_DIR}/**/*.laz", recursive=True))
if laz_files:
    plot_point_cloud(
        laz_files[0], color="rgb", max_points=150_000,
        title="SF LiDAR — raw returns (one EPT node, RGB-colored)",
    )
else:
    print(f"No .laz staged under {LAZ_DIR}; point cloud preview skipped.")


In [ ]:
from databricks.labs.gbx.sample.lidar import lonlat_to_3857, aoi_lonlat_to_3857
X0, Y0, X1, Y1 = aoi_lonlat_to_3857(*SF_AOI)

if DEMO:
    _gx, _gy = lonlat_to_3857(-122.486, 37.769)   # Golden Gate Park centroid
    _tx, _ty = int((_gx - X0) / TILE_M), int((_gy - Y0) / TILE_M)
    DEMO_FILTER = (
        F.col("tx").between(_tx - 2, _tx + 2) &
        F.col("ty").between(_ty - 2, _ty + 2)
    )
    dtm_in = dtm.where(DEMO_FILTER)
    dsm_in = dsm.where(DEMO_FILTER)
    chm_in = chm.where(DEMO_FILTER)
    print(f"DEMO mode: {dtm_in.count()} DTM | {dsm_in.count()} DSM | {chm_in.count()} CHM tiles "
          f"in ±2 km window around Golden Gate Park (tx={_tx}, ty={_ty})")
else:
    dtm_in, dsm_in, chm_in = dtm, dsm, chm


## Part 3 — T2: DEM / DSM surfaces → elevation isobands → H3 cells

Turns the EPSG:3857 DTM and DSM surfaces from Part 2 into elevation-band H3 cell tables
(`wc_h3_dem`, `wc_h3_dsm`). Key steps:

1. **Land mask** (Overture water features → SF land polygon) — clips Bay and Pacific
   pixels to NoData before isoband extraction.
2. **CRS transform**: `rst_transform(tile, 4326)` — `h3_try_coverash3` requires lon/lat WKB.
3. **`rst_clip`** — apply land mask to each reprojected tile.
4. **`rst_isoband`** — extract contiguous elevation-band polygons.
5. **`h3_try_coverash3`** — index each band polygon to H3 cells (no driver-side indexing).
6. **Persist** both tables; print counts and band distribution.
7. **Cumulative tiers** and a static band map.

In [ ]:
# --- Land mask: Overture base/water → SF land polygon ---
# Subtract Bay and Pacific water features from the AOI box to get a land multipolygon.
# Used as the rst_clip cutline to set water/ocean pixels to NoData before isoband.
ov = OvertureClient()
if not glob.glob(f"{LAND_DIR}/**/*.parquet", recursive=True):
    assets = ov.discover(SF_AOI, themes=["base"]).filter(F.col("type") == "water")
    ov.download(assets, LAND_DIR, bbox=SF_AOI)
water = ov.read(LAND_DIR, theme="base", type="water", bbox=SF_AOI)
_wg = [_wkb.loads(bytes(r["geometry"])) for r in water.select("geometry").collect()]
W, S, E, N = SF_AOI
land_geom = _box(W, S, E, N).difference(unary_union(_wg)) if _wg else _box(W, S, E, N)
land_wkb  = land_geom.wkb     # EPSG:4326 land multipolygon (WKB) — cutline for rst_clip
print(f"Land mask: subtracted {len(_wg)} water feature(s); "
      f"land area \u2248 {land_geom.area / _box(W,S,E,N).area:.2f} of AOI")


In [ ]:
# --- Elevation-band H3 cells: EPSG:3857 surface tiles → H3 cells (EPSG:4326) ---
# breaks_arr is broadcast to every executor so each call to rst_isoband uses the
# same set of elevation thresholds.
breaks_arr = F.array(*[F.lit(b) for b in BREAKS_M])


def _surface_to_h3_cells(surface_df, tile_col):
    """EPSG:3857 surface tiles → elevation isobands → H3 cells (EPSG:4326).

    Steps:
      1. rst_transform(tile, 4326)  — REQUIRED: h3_try_coverash3 needs lon/lat WKB.
      2. rst_clip(land_wkb)         — mask Bay and Pacific to NoData before isoband.
      3. rst_isoband(breaks)        — produces (band_level, geom_wkb, elev_lo, elev_hi).
      4. h3_try_coverash3(geom_wkb, res) + explode — solid, gap-free H3 cells per band.
    """
    # Step 1: reproject 3857 → 4326 (lon/lat required for H3 product functions).
    t4326 = surface_df.select(
        "tx", "ty",
        rx.rst_transform(tile_col, F.lit(4326)).alias("tile_4326"),
    )
    # Step 2: clip to land polygon (removes Bay/Pacific pixels before isoband).
    clipped = t4326.select(
        "tx", "ty",
        rx.rst_clip("tile_4326", F.lit(land_wkb), F.lit(True)).alias("tile"),
    )
    # Step 3: extract contiguous elevation-band polygons.
    patches = (
        clipped
        .select(F.explode(rx.rst_isoband("tile", breaks_arr)).alias("p"))
        .select(
            F.col("p.band").alias("band_level"),
            F.col("p.geom_wkb").alias("geom_wkb"),
            F.col("p.lower").alias("elev_lo"),
            F.col("p.upper").alias("elev_hi"),
        )
    )
    # Step 4: H3 index via Databricks product h3_try_coverash3 (WKB input, no ST_ wrap).
    cells = (
        patches
        .select(
            "band_level", "elev_lo", "elev_hi",
            F.explode(
                DBF.h3_try_coverash3(F.col("geom_wkb"), F.lit(H3_RESOLUTION))
            ).alias("cellid"),
        )
        .where(F.col("cellid").isNotNull())
        .distinct()
        .withColumn("res", F.lit(H3_RESOLUTION))
    )
    return cells.select("cellid", "res", "band_level", "elev_lo", "elev_hi")


In [ ]:
# --- Isoband validation: assert >=3 non-empty elevation bands on a sample ---
# Runs before the full DEM/DSM passes to catch a misconfigured BREAKS_M or a
# mismatched CRS (e.g. tiles still in EPSG:3857 — all elevations would be outside
# the 0-300 m range and yield 0 bands). Sample 5 tiles so the elevation spread
# is enough to span at least 3 bands even in the flat DEMO AOI.
sample_tile = dtm_in.limit(5)
sample_cells = _surface_to_h3_cells(sample_tile, "dtm")
band_counts = sample_cells.groupBy("band_level", "elev_lo", "elev_hi").count().orderBy("band_level")
band_counts.show()
n_nonempty = band_counts.count()
assert n_nonempty >= 3, (
    f"Expected ≥3 non-empty elevation bands, got {n_nonempty}. "
    "Check BREAKS_M range vs actual DTM elevation distribution."
)
print(f"Band validation: {n_nonempty} non-empty elevation bands — OK")

In [ ]:
# --- Produce and persist wc_h3_dem (bare earth DTM elevation bands) ---
dem_cells = _surface_to_h3_cells(dtm_in, "dtm")   # DEM = bare earth DTM
dem_h3 = _persist(dem_cells, "dem")                # → geospatial_docs.geobrix.wc_h3_dem
print(f"wc_h3_dem: {dem_h3.count()} rows (H3 res {H3_RESOLUTION})")
dem_h3.groupBy("band_level", "elev_lo", "elev_hi").count().orderBy("band_level").show()


In [ ]:
# --- Produce and persist wc_h3_dsm (surface incl. buildings/canopy) ---
dsm_cells = _surface_to_h3_cells(dsm_in, "dsm")   # DSM = surface incl. buildings/canopy
dsm_h3 = _persist(dsm_cells, "dsm")               # → geospatial_docs.geobrix.wc_h3_dsm
print(f"wc_h3_dsm: {dsm_h3.count()} rows (H3 res {H3_RESOLUTION})")


In [ ]:
# --- DEM cumulative tiers: tier T = 'elevation >= BREAKS_M[T]' ---
# A cell in interval band i belongs to tiers 0..i; coverage depth climbs with
# terrain height — the telco nested-coverage model applied to elevation.
dem_tiers = (
    dem_h3
    .select("cellid", "res",
            F.explode(F.sequence(F.lit(0), F.col("band_level"))).alias("tier"))
    .distinct()
)
dem_tiers.groupBy("tier").count().orderBy("tier").show()


In [ ]:
# --- Visualize DEM H3 cells: dissolve by band → <=10 footprints, fast render ---
dem_gdf = cells_as_gdf(
    dem_h3, "cellid",
    extra_cols=["band_level"],
    dissolve_by="band_level",
    dissolve_engine="product",
)
plot_static(
    dem_gdf, column="band_level", cmap="terrain",
    title=f"DEM (bare earth) — elevation bands, H3 res {H3_RESOLUTION}",
)


## CHM → H3 max canopy height

The `gbx_rst_h3_rastertogridmax` UDTF aggregates all raster pixels whose centroid falls in
each H3 cell and returns the maximum value per cell. Applied to the CHM (canopy height in
metres), it gives the tallest vegetation or structure per H3 hexagon — the surface height
signal that nb4 will use for line-of-sight analysis.

**Scale note:** `DEMO=True` (default) runs on ~25 tiles (Golden Gate Park ±2 km); the full
San Francisco AOI is 336 CHM tiles (~336 M pixels at 1 m resolution). At res 9 over full SF,
expect 5–15 min on Serverless — profile with `DEMO=True` first and set `DEMO=False` when
the output is validated.


In [ ]:
# rst_h3_rastertogridmax (and all H3 UDTFs) require lon/lat EPSG:4326 tiles.
chm_4326 = chm_in.select(
    "tx", "ty",
    rx.rst_transform("chm", F.lit(4326)).alias("chm_4326"),
)
# Register as a temp view for the SQL LATERAL call below.
# Serverless: repartition(N, column) for fan-out parallelism (sparkContext unavailable).
chm_4326.repartition(64, "tx") \
        .createOrReplaceTempView("chm_4326_tiles")
print(f"CHM tiles prepared for H3 rastertogridmax: {chm_4326.count()} tile(s) at EPSG:4326")


In [ ]:
# SQL LATERAL UDTF (ANSI form): invoke via spark.sql — the Python wrapper raises NotImplementedError.
# VERIFIED output schema (T0, live cluster): (band INT, cellID LONG, measure DOUBLE); band=1 is the
# single CHM band. Do NOT use `LATERAL VIEW OUTER ... AS a,b,c` — positional renaming mislabels the
# columns (band->a, cellID->b, measure->c). Use ANSI LATERAL with the real column names:
chm_cells_raw = spark.sql(f"""
    SELECT t.cellID AS cellid, t.measure AS chm_z
    FROM   chm_4326_tiles,
           LATERAL gbx_rst_h3_rastertogridmax(chm_4326, {H3_RESOLUTION}) t
    WHERE  t.band = 1
""")
# Multiple tiles may map the same cellid at tile boundaries → take max across tiles.
chm_cells = (
    chm_cells_raw
    .groupBy("cellid").agg(F.max("chm_z").alias("max_chm_z"))
    .withColumn("res", F.lit(H3_RESOLUTION))
    .select("cellid", "res", "max_chm_z")
)
print(f"CHM H3 cells (before persist): {chm_cells.count()}")
chm_cells.selectExpr("min(max_chm_z)", "max(max_chm_z)", "avg(max_chm_z)").show()


In [ ]:
chm_h3 = _persist(chm_cells, "chm")   # → geospatial_docs.geobrix.wc_h3_chm
print(f"wc_h3_chm: {chm_h3.count()} cells | max_chm_z: "
      f"[{chm_h3.selectExpr('min(max_chm_z)').first()[0]:.1f} m, "
      f"{chm_h3.selectExpr('max(max_chm_z)').first()[0]:.1f} m]")


In [ ]:
chm_gdf = cells_as_gdf(
    chm_h3, "cellid",
    extra_cols=["max_chm_z"],
    dissolve_engine="product",
)
plot_static(
    chm_gdf, column="max_chm_z", cmap="viridis",
    title=f"CHM — max canopy height per H3 cell (res {H3_RESOLUTION})",
)
